In [18]:
# libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

In [19]:
# load csv
file_path = "state_ofthe_union_texts.csv"
if os.path.exists(file_path):
    df_existing = pd.read_csv(file_path)
else:
    df_existing = pd.DataFrame(columns=['President', 'Year', 'Title', 'Text'])

In [20]:
# Speech URL - this is where I change it 
url = 'https://www.presidency.ucsb.edu/documents/address-before-joint-session-the-congress-the-state-the-union-3'
res = requests.get(url)
soup = BeautifulSoup(res.content, 'html.parser', from_encoding='utf-8')

In [22]:
# Scraping! 

# Get President
president_elem = soup.select_one('.diet-title a')
president = president_elem.get_text(strip=True) if president_elem else "Unknown"

# Get Year
date_elem = soup.select_one('.field-docs-start-date-time')
year = pd.to_datetime(date_elem.text.strip()).year if date_elem else None

# Get Title
title_elem = soup.select_one('h1.page-title')
title = title_elem.get_text(strip=True) if title_elem else "No Title Found"

# Get Speech Text (preserve paragraph breaks)
speech_elem = soup.select_one('.field-docs-content')
if speech_elem:
    paragraphs = speech_elem.find_all('p')
    speech_text = '\n\n'.join(p.get_text(strip=True) for p in paragraphs)
else:
    speech_text = "No speech text found."

In [23]:
# add to csv

# Create and append new row
new_row = {
    'President': president,
    'Year': year,
    'Title': title,
    'Text': speech_text
}

df_existing = pd.concat([df_existing, pd.DataFrame([new_row])], ignore_index=True)

# Save to CSV
df_existing.to_csv('state_ofthe_union_texts_updatedRR.csv', index=False)